### Apply the same criteria for wv7 to process LLM responses (summary)
- Check whether LLM responses are valid (in between the range)
- Revise LLM item responses to be consistent in sentiment/intensity direction
    - weak/negative --> strong/positive
- Apply MinMaxScaler
    - Use the raw scale from survey design
- Apply the same process for Claude and deepseek responses
    - Invalid check
    - Reverse to make responses consistent
    - Scale to 0~1


In [1]:
import json
import pandas as pd 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from matplotlib import pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df_gpt_raw = pd.read_csv("./data/gpt5_125.csv", index_col="Unnamed: 0")
df_gpt_raw.index.name = "Country"

((67, 125), (85, 162))

In [3]:
df_gpt_raw

,A001,A002,A003,A004,A005,A006,A008,A009,A062,A165,...,H001,H006_01,H006_02,H008_01,H008_02,H008_03,H008_04,H008_07,H008_08,H008_09
Country,,,,,,,,,,,,,,,,,,,,,
Andorra,1,2,2,3,2,3,1,2,2,1,...,1,3,3,4,4,4,4,1,2,4
Argentina,1,1,2,2,2,3,2,2,1,2,...,3,1,2,3,3,3,3,2,2,4
Armenia,1,2,2,2,1,2,2,2,1,2,...,3,2,1,3,4,3,3,2,2,4
Australia,1,1,1,3,2,4,2,2,2,1,...,2,2,2,4,4,4,4,2,1,4
Bangladesh,1,2,2,2,1,1,2,2,1,2,...,2,1,1,2,3,2,2,2,2,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Uzbekistan,1,2,2,3,1,2,2,2,2,2,...,2,2,1,4,4,4,4,2,2,4
Venezuela,1,1,2,2,1,2,3,3,1,2,...,3,1,1,2,2,2,2,2,2,3
Vietnam,1,2,2,3,1,3,2,2,2,2,...,2,2,1,4,3,3,3,2,2,4


In [5]:
def check_invalid(df_data):
    """
    check whether there is negative or missing values 
    """
    numeric_cols = df_data.select_dtypes(include="number").columns
    has_missing_or_negative = df_data[numeric_cols].lt(0) | df_data[numeric_cols].isna()
    cols_with_missing_or_negative = has_missing_or_negative.any()
    cols_with_missing_or_negative = cols_with_missing_or_negative[cols_with_missing_or_negative]

    print(f"{len(cols_with_missing_or_negative)} out of {len(numeric_cols)} numeric columns contain missing or negative values")
    cols_with_missing_or_negative

In [6]:
check_invalid(df_gpt_raw)

0 out of 125 numeric columns contain missing or negative values


### Check whether the item responses are valid 
- load the scaler

In [9]:
df_country_mean = pd.read_csv("../data/processed/wvs7_country_avg.csv").set_index('Country')
selected_items  = list(df_country_mean.columns)
df_country_mean.shape

(66, 117)

In [11]:
set(selected_items).issubset(set(df_gpt_raw.columns))

True

In [12]:
df_country_mean.head()

,A001,A002,A003,A004,A005,A006,A008,A009,A062,A165,...,H001,H006_01,H006_02,H008_01,H008_02,H008_03,H008_04,H008_07,H008_08,H008_09
Country,,,,,,,,,,,,,,,,,,,,,
Andorra,0.961487,0.821381,0.859562,0.353293,0.824684,0.344977,0.763280,0.772410,0.414796,0.255744,...,0.904287,0.516966,0.549432,0.044245,0.032934,0.033966,0.145563,0.662338,0.366000,0.036703
Argentina,0.969425,0.819541,0.730063,0.397602,0.843865,0.535354,0.729400,0.711538,0.374747,0.206933,...,0.524476,0.492355,0.474400,0.096141,0.372294,0.133667,0.191049,0.641642,0.364729,0.066667
Armenia,0.982284,0.746318,0.698951,0.429276,0.885246,0.866831,0.771091,0.565057,0.633005,0.081081,...,0.532125,0.646848,0.731397,0.130553,0.047893,0.173508,0.284739,0.447547,0.178037,0.038189
Australia,0.963209,0.840128,0.784548,0.529389,0.668203,0.367424,0.739058,0.736219,0.474972,0.540179,...,0.663887,0.415698,0.393101,0.055370,0.177506,0.101764,0.145717,0.767468,0.564278,0.017065
Bangladesh,0.995000,0.643813,0.602747,0.307890,0.925802,0.977444,0.721667,0.678750,0.329163,0.129274,...,0.716152,0.544275,0.756312,0.173901,0.127197,0.352810,0.413861,0.740151,0.274707,0.099659


In [ ]:
df_question_scale = pd.read_csv("./data/question_scale_ranges.csv")
df_question_scale.head()

,min,max,WVS7,Variable
0,1,4,Q1,A001
1,1,4,Q2,A002
2,1,4,Q3,A003
3,1,4,Q4,A004
4,1,4,Q5,A005


Check whether llm item responses are valid (in between the range)

In [ ]:
# for each item in df_gpt, check whether its values are within [min, max] from df_question_scale
df_gpt = df_gpt_raw[selected_items]

def check_range(df_data):

    scale_lookup = df_question_scale.set_index("Variable")[["min", "max"]]

    out_of_range = pd.DataFrame(index=df_data.index)
    for item in selected_items:
        item_min = scale_lookup.loc[item, "min"]
        item_max = scale_lookup.loc[item, "max"]
        out_of_range[item] = ~df_data[item].between(item_min, item_max)

    items_out_of_range = out_of_range.any()
    items_out_of_range = items_out_of_range[items_out_of_range]

    print(f"{len(items_out_of_range)} out of {len(selected_items)} items have values outside their [min, max] range")
    items_out_of_range

check_range(df_gpt)

0 out of 117 items have values outside their [min, max] range


### Revise items responses to be consistent
- weak/negative/low --> strong/positive/high 

In [15]:
df_question_scale[df_question_scale['Variable'] == 'A165']

,min,max,WVS7,Variable
148,1,2,Q57,A165


In [16]:
# original responses, before revision
df_gpt[['H008_07', 'A165', 'A173', 'A170']].head()

,H008_07,A165,A173,A170
Country,,,,
Algeria,2,2,4,5
Andorra,1,1,7,8
Argentina,2,2,5,5
Armenia,2,2,4,6
Australia,2,1,7,7


In [17]:
# reverse item positive responses: strong (small value) to weak (large value)
def reverse(df_data):

    retain_items = ["A173", "A170", "C006", "E268", "G052", "F114A", "F114B", "F116", "F117", "F120", "F121", "F122", "F123", "F114C", "F114D", "F144_02", "E224", "E225", "E226", "E227", "E229", "E233A", "E233B", "E233", "E235", "E236"]

    scale_lookup = df_question_scale.set_index("Variable")[["min", "max"]]

    for item in set(selected_items).difference(set(retain_items)):
        item_min = scale_lookup.loc[item, "min"]
        item_max = scale_lookup.loc[item, "max"]
        df_data[item] = df_data[item].where(df_data[item] < 0, item_min + item_max - df_data[item])

reverse(df_gpt)

In [18]:
# response after revision
df_gpt[['H008_07', 'A165', 'A173', 'A170']].head()

,H008_07,A165,A173,A170
Country,,,,
Algeria,1,1,4,5
Andorra,2,2,7,8
Argentina,1,1,5,5
Armenia,1,1,4,6
Australia,1,2,7,7


### Apply minmaxscaler to LLM responses

In [19]:
from sklearn.preprocessing import MinMaxScaler

def scale(df_data):
    scale_lookup = df_question_scale.set_index("Variable")[["min", "max"]]

    for item in selected_items:
        item_min = scale_lookup.loc[item, "min"]
        item_max = scale_lookup.loc[item, "max"]
        scaler = MinMaxScaler()
        scaler.fit([[item_min], [item_max]])
        df_data[item] = scaler.transform(df_data[[item]])

scale(df_gpt)

In [20]:
df_gpt.head()

,A001,A002,A003,A004,A005,A006,A008,A009,A062,A165,...,H001,H006_01,H006_02,H008_01,H008_02,H008_03,H008_04,H008_07,H008_08,H008_09
Country,,,,,,,,,,,,,,,,,,,,,
Algeria,1.0,1.000000,0.666667,0.666667,1.000000,1.000000,0.666667,0.75,1.0,0.0,...,0.666667,1.000000,1.000000,0.333333,0.333333,0.333333,0.666667,0.0,0.0,0.0
Andorra,1.0,0.666667,0.666667,0.333333,0.666667,0.333333,1.000000,0.75,0.5,1.0,...,1.000000,0.333333,0.333333,0.000000,0.000000,0.000000,0.000000,1.0,0.0,0.0
Argentina,1.0,1.000000,0.666667,0.666667,0.666667,0.333333,0.666667,0.75,1.0,0.0,...,0.333333,1.000000,0.666667,0.333333,0.333333,0.333333,0.333333,0.0,0.0,0.0
Armenia,1.0,0.666667,0.666667,0.666667,1.000000,0.666667,0.666667,0.75,1.0,0.0,...,0.333333,0.666667,1.000000,0.333333,0.000000,0.333333,0.333333,0.0,0.0,0.0
Australia,1.0,1.000000,1.000000,0.333333,0.666667,0.000000,0.666667,0.75,0.5,1.0,...,0.666667,0.666667,0.666667,0.000000,0.000000,0.000000,0.000000,0.0,1.0,0.0


In [ ]:
# df_gpt.to_csv("./data/gpt_scaled.csv")


### Apply the same process for Claude and deepseek
- invalid check
- reverse to make responses consistent
- scale

In [ ]:
df_claude_raw = pd.read_csv("./data/claude_125.csv", index_col="Unnamed: 0")
df_claude_raw.index.name = "Country"

df_deepseek_raw = pd.read_csv("./data/deepseek_125.csv", index_col="Unnamed: 0")
df_deepseek_raw.index.name = "Country"

df_claude_raw.shape, df_deepseek_raw.shape

((67, 125), (67, 125))

In [ ]:
df_claude_raw.head()

,A001,A002,A003,A004,A005,A006,A008,A009,A062,A165,...,H001,H006_01,H006_02,H008_01,H008_02,H008_03,H008_04,H008_07,H008_08,H008_09
Country,,,,,,,,,,,,,,,,,,,,,
Andorra,1,1,2,3,2,2,1,2,2,1,...,1,3,2,4,4,4,4,1,1,4
Argentina,1,1,2,2,2,3,2,2,1,2,...,3,2,2,3,2,3,2,1,1,3
Armenia,1,1,2,2,2,1,2,2,2,2,...,3,2,2,3,3,3,2,2,2,3
Australia,1,1,2,3,2,4,2,2,2,1,...,2,3,2,4,3,4,3,1,1,4
Bangladesh,1,1,3,3,2,1,2,2,2,2,...,3,2,1,3,3,2,2,2,2,3


Data processing for claude responses

In [ ]:
df_claude = df_claude_raw[selected_items]
check_range(df_claude)
reverse(df_claude)
scale(df_claude)
# df_claude.to_csv("./data/claude_scaled.csv")

0 out of 117 items have values outside their [min, max] range


Data processing for deepseek responses

In [ ]:
df_deepseek = df_deepseek_raw[selected_items]
check_range(df_deepseek)
reverse(df_deepseek)
scale(df_deepseek)
# df_deepseek.to_csv("./data/deepseek_scaled.csv")


0 out of 117 items have values outside their [min, max] range


------------------------------the end------------------------------